# Test Data generation & Model predctions

In [ ]:
import json, os, random, torch
import numpy as np
import importlib

import Data.IO as dio
from Data.Generators import generateTransforms, getPatches
from utils import load_model
from nibabel.affines import apply_affine

import Dataset #import YoloDataset
import Volume.Patch as vp
import torch.optim as optim
import losses
from tqdm import tqdm

In [ ]:
# load pretrained model
train_dir = "/path/to/0Work/Experiments/5Folds_Nancy/Fold1/"
with open(os.path.join(train_dir, 'ndl_config.json'),'r') as f:
    config = json.load(f)

_, _, patients = dio.readSplit(config['split_file'])
patients = sorted(patients)[:1]
#patients =['/path/to/Data/CHRU/P0002']
print(patients)

model, epoch = load_model(config, 'cuda', last=True)

#from model import UNetYOLO
#model = UNetYOLO(num_classes = 0, params_per_box = 4, anchors_per_scale = 3)
#model, _ = load_model(config, logger=None, device="cpu", last=True)
model.eval(); model.to('cuda');

In [ ]:
def tt(train_patches, patient_name, model=None):
    cmp = 1
    for p in train_patches:
        if p['status']:
            affine, disp = None, None#generateTransforms(trans, scal, rot, p['point'], disp)              
            v, t, vessel, v2m = vp.getPatchAndTruth(vol = p['data'], 
                                               truth = p['truth'],
                                               vox2met = p['vox2met'], 
                                               center = p['point'], 
                                               size = 0.4*48, 
                                               dim = 48, 
                                               aneurysms=p['aneurysms'], 
                                               affine=None, 
                                               disp=None,  
                                               vessel=p["vessel"], 
                                               refined = True,
                                               return_bbox=None)
#            print(bbox)
            if model is not None:
                output = model(torch.from_numpy(v[np.newaxis][np.newaxis]).float().to('cuda'))[-1]
                output = np.squeeze(output.detach().cpu().numpy())#.astype('float64')
                print(output.min(), output.max())
            if patient_name is not None:
                dio.save_nii_to_file(f"/path/to/Patches/{patient_name}_v_{cmp}.nii.gz", v, v2m)
                dio.save_nii_to_file(f"/path/to/Patches/{patient_name}_t_{cmp}.nii.gz", t, v2m)
                dio.save_nii_to_file(f"/path/to/Patches/{patient_name}_o_{cmp}.nii.gz", output, v2m)
                

            cmp+=1

In [ ]:
main_dir = "/path/to/Data/CHRU/"
patients = os.listdir(main_dir)[:1]
#patients = ["P0019" ]
patients = [os.path.join(main_dir, f) for f in patients if '.' not in f]
print(len(patients))

In [ ]:
train_db = dio.add_points_to_patient_data(dio.read_patient_data_base(  pat_list=patients, 
                                                                       volume='init volume', 
                                                                       normalize=None, 
                                                                       vessel=None, 
                                                                       loadtruthFromFile=False,
                                                                       workers= 1
                                                                    ),
                                             'points.csv',
                                             nb=0)
train_patches = getPatches(train_db, pos_dup=1, loadTruthFromFile=False)

In [ ]:
tt(train_patches, patient_name="P0019", model=model)

In [ ]:
importlib.reload(YoloDataset)
# YOLO Hyperparams
anchors = [[0.4791667, 0.31083333, 0.17],
           [0.11456989,  0.1 , 0.08416667 ],
           [0.04333333, 0.01625,  0.0040]]
S = [6, 12, 24]

dataset = YoloDataset.Dataset (
                    patches = train_patches,
                    size = [x * y for x, y in zip(config["patch_size"], config["patch_shape"])],
                    dim = config["patch_shape"],
                    neg_trans = config["negative sample shift"],
                    neg_rot = config["negative sample rotation"],
                    neg_disp = config["negative sample distortion"],
                    neg_scal = config["negative sample scaling"],
                    pos_trans = config["positive sample shift"],
                    pos_rot = config["positive sample rotation"],
                    pos_disp = config["positive sample distortion"],
                    pos_scal = config["positive sample scaling"],
                    flip_prob = config['flip_prob'],
                    refined = True,
                    anchors = anchors,
                    scales = S,
                    num_params=4,
                    nb_classes=0)
loader = torch.utils.data.DataLoader(dataset=dataset, batch_size=2, shuffle=False)

# visualize truth and prediciton in 3D space

In [ ]:
import matplotlib.pyplot as plt
%matplotlib widget

def cells_to_spheres(predictions, obj_thresh, input_shape, anchors, is_pred=True):
    nb_anchors = anchors.shape[0]
    spheres = []
    if is_pred:
        predictions = predictions.numpy()
        anchors = anchors.reshape(len(anchors), 1, 1, 1, 1)
        predictions[..., :3] = torch.sigmoid(torch.from_numpy(predictions[..., :3])).numpy()
        predictions[..., 3:4] = (torch.exp(torch.from_numpy(predictions[..., 3:4])) * anchors).numpy()
        predictions[..., 4:] = torch.sigmoid(torch.from_numpy(predictions[..., 4:])).numpy()
        
    grid_h, grid_w, grid_d = predictions.shape[1:4]
    for i in range(grid_h):
        for j in range(grid_w):
            for k in range(grid_d):
                row, col, dep = i, j, k
                for b in range(nb_anchors): # scales=anchors
                    if is_pred :
                        confidence = predictions[b, i, j, k, 4]
                    else:
                        confidence = predictions[b, i, j, k, 4]

                    if(confidence >= obj_thresh):
                        x, y, z, radius = predictions[b, i, j, k, :4]
                        x = (row + x) * (input_shape[0]/grid_h)
                        y = (col + y) * (input_shape[1]/grid_w)
                        z = (dep + z) * (input_shape[2]/grid_d)
                        radius = radius * (input_shape[0]/grid_h)
                        spheres.append([x, y, z, radius, confidence])
    return spheres


def spheres_to_metric (spheres, v2m):
    """   Go from voxel to mm space   """
    def apply_transormation(elements, trans):
        return apply_affine (trans, elements)
    v2m = v2m.squeeze()
    assert isinstance(spheres, list)   
    vox_size = np.linalg.norm(v2m[:3,:3], axis=0) [0]
    for sphere in spheres:
        sphere[:3] = apply_transormation(np.array(sphere[:3]), v2m)
        sphere[3] = float(sphere[3] * vox_size)
    return spheres

def test_data_generation(loader, anchors, S, model=None):
    scaled_anchors = (torch.tensor(anchors).reshape(3, 3, 1) * torch.tensor(S).unsqueeze(1).unsqueeze(1).repeat(1, 3, 1) )
    for batch in loader:
        vol, truth, truth_spheres, v2m = batch['volume'], batch['truth_volume'], batch['truth_boxes'], batch['v2m']
        boxes_truth = []
        nb_anchors = truth_spheres[0].shape[1]
        for i in range(nb_anchors):
            anchor = scaled_anchors[i]
            boxes_truth += cells_to_spheres(truth_spheres[i].squeeze(), obj_thresh=0.7, input_shape=[48,48,48], anchors=anchor, is_pred=False)
        boxes_truth = non_max_suppression(boxes_truth, iou_threshold=1)
        boxes_truth = spheres_to_metric (boxes_truth, v2m)
        
        boxes_pred = []
        if model is not None:
            model.eval()
            with torch.no_grad():
                _, pred_spheres = model(vol)
            boxes_pred = []
            for i in range(nb_anchors):
                anchor = scaled_anchors[i]
                boxes_pred += cells_to_spheres(pred_spheres[i].squeeze(), obj_thresh=0.5, anchors=anchor, input_shape=[48,48,48], is_pred=True)
            boxes_pred = non_max_suppression(boxes_pred, iou_threshold=0.5)
            boxes_pred = spheres_to_metric(boxes_pred, v2m=v2m)
            print(boxes_pred)
        return
        triple_sphere_plot (boxes_pred, boxes_truth)

def ellipse_xyz(center, extent):
    [a, b, c] = extent
    u, v = np.mgrid[0:2 * np.pi:80j, 0:np.pi:40j]
    x = a * np.cos(u) * np.sin(v) + center[0]
    y = b * np.sin(u) * np.sin(v) + center[1]
    z = c * np.cos(v) + center[2]
    return x, y, z

def triple_sphere_plot(preds, truth=None):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    if preds is not None:
        for (sample, c) in zip(preds, ['r', 'g', 'b']):
            x, y, z = ellipse_xyz(sample[:3], np.ones([3]) * sample[3])
            ax.plot_wireframe(x, y, z, color=c)
    if truth is not None:
        x, y, z = ellipse_xyz(truth[0][:3], np.ones([3]) * truth[0][3])
        ax.plot_wireframe(x, y, z, color='black')

test_data_generation(loader, anchors, S, model=model)

# Training UNetYolo model

In [ ]:
importlib.reload(losses)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss = losses.UNetYoloLoss(nb_anchors_per_scale=3)
optimizer = optim.Adam( model.parameters(), lr=1e-4, weight_decay=1e-5)

scaled_anchors = (torch.tensor(anchors).reshape(3, 3, 1) * torch.tensor(S).unsqueeze(1).unsqueeze(1).repeat(1, 3, 1) ).to(device)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
model.to(device)
model.train();

loop = tqdm(range(50), leave=True)
for epoch in loop:
    avg_loss = []
    for batch in loader:
        vol, truth, truth_spheres, v2m = batch['volume'], batch['truth_volume'], batch['truth_boxes'], batch['v2m']
        with torch.cuda.amp.autocast():
            pred_vol, pred_spheres = model (vol.to(device))
            loss_tot = loss(inputs_vol=pred_vol[-1], inputs_boxes=pred_spheres, targets_vol=truth.to(device), targets_boxes=truth_spheres, anchors=scaled_anchors)
        optimizer.zero_grad()
        scaler.scale(loss_tot).backward()
        scaler.step(optimizer)
        scaler.update()
        avg_loss.append(loss_tot.item())
    loop.set_description(f'Epoch {epoch} : Loss= {round(sum(avg_loss)/len(avg_loss), 3)}')
        #break


In [ ]:
detection_thr = 0.5
model.to('cpu')
model.eval()
cmp = 0
max_per_patch = 4

test_loader = torch.utils.data.DataLoader(dataset=dataset, batch_size=1, shuffle=False)
for batch in test_loader:
    patch, truth, truth_spheres, v2m = batch['volume'],batch['truth_volume'], batch['truth_boxes'], batch['v2m']
    with torch.no_grad():
        pred_vol, pred_spheres = model (patch)
    spheres = []
    
    # for ground truth check
    #for i in range(len(truth_spheres)):
    #    spheres_scale = cells_to_spheres(truth_spheres[i].squeeze().detach().numpy(), obj_thresh=detection_thr, anchors=scaled_anchors[i].cpu(), input_shape=patch.shape[-3:], is_pred=False)
    #    spheres_scale = spheres_to_metric(spheres_scale, v2m=v2m[0].numpy())
    #    spheres.extend(spheres_scale)
    #spheres = non_max_suppression(bboxes=spheres, iou_threshold=0.01)
    
    # for inference (arg: is_pred = True)
    for i in range(len(pred_spheres)):
        spheres_scale = cells_to_spheres(pred_spheres[i].squeeze().detach().numpy(), obj_thresh=detection_thr, anchors=scaled_anchors[i].cpu(), input_shape=patch.shape[-3:], is_pred=True)
        spheres_scale = spheres_to_metric(spheres_scale, v2m=v2m[0].numpy())
        spheres.extend(spheres_scale)
    spheres = non_max_suppression(bboxes=spheres, iou_threshold=0.01) [:max_per_patch]
    print(spheres)
    if len(spheres) > 0:
        dio.save_nii_to_file (f"/path/to/Test/{cmp}_vol.nii.gz", patch[0][0].numpy(), v2m[0].numpy())
        dio.save_nii_to_file (f"/path/to/Test/{cmp}_truth.nii.gz", truth[-1][0].numpy(), v2m[0].numpy())
    cmp += 1
    break


In [ ]:
# Do Evaluation
def get_evaluation_spheres(
    loader,
    model,
    iou_threshold,
    anchors,
    threshold,
    device="cuda",
):
    # make sure model is in eval before get bboxes
    model.eval()
    model = model.to(device)
    train_idx = 0
    all_pred_boxes = []
    all_true_boxes = []
        
    for batch_idx, batch in enumerate(tqdm(loader)):
        x, truth, truth_spheres = batch['volume'],batch['truth_volume'], batch['truth_boxes']
        x = x.to(device)
        with torch.no_grad():
            _, predictions = model (x)

        batch_size = x.shape[0]
        assert batch_size == 1
        bboxes = [[] for _ in range(batch_size)]
        for i in range(3):
            S = predictions[i].shape[2]
            anchor = anchors[i].to(device)
            print(x.shape[-3:])
            cells_to_spheres(predictions[i].cpu().numpy(), obj_thresh=threshold, input_shape=x.shape[-3:], anchors=anchor.cpu().numpy())
            return
            boxes_scale_i = cells_to_bboxes(
                predictions[i], anchor, S=S, is_preds=True
            )
            for idx, (box) in enumerate(boxes_scale_i):
                bboxes[idx] += box

        # we just want one bbox for each label, not one for each scale
        true_bboxes = cells_to_spheres(predictions, obj_thresh, input_shape, anchors)
#        true_bboxes = cells_to_bboxes( labels[2], anchor, S=S, is_preds=False)

        for idx in range(batch_size):
            nms_boxes = non_max_suppression(
                bboxes[idx],
                iou_threshold=iou_threshold,
                threshold=threshold,
                box_format=box_format,
            )

            for nms_box in nms_boxes:
                all_pred_boxes.append([train_idx] + nms_box)

            for box in true_bboxes[idx]:
                if box[1] > threshold:
                    all_true_boxes.append([train_idx] + box)

            train_idx += 1

    model.train()
    return all_pred_boxes, all_true_boxes

get_evaluation_spheres(loader=test_loader, model=model, iou_threshold=0.01, anchors=scaled_anchors, threshold=detection_thr,  device="cuda")

## Test Yolo truth generation

In [ ]:
#spheres = cells_to_spheres(targets[0], obj_thresh=0.5, anchors[0], input_shape=[48, 48, 48])


#spheres_to_metric(sphres)      
def non_max_suppression(bboxes, iou_threshold):
    def intersection_over_union(prediction, truth):
        def get_intersection(point1, point2):
            pos1, pos2 = point1[:3], point2[:3]
            r1, r2 = point1[3], point2[3]
            d = sum((pos1 - pos2) ** 2) ** 0.5
            # check they overlap
            if d >= (r1 + r2):
                return 0.
            # check if one entirely holds the other
            elif d == 0 and min(r1, r2) > 0:
                return 4. / 3. * np.pi * min(r1, r2) ** 3
            elif r1 > (d + r2):  # 2 is entirely contained in one
                return 4. / 3. * np.pi * r2 ** 3
            elif r2 > (d + r1):  # 1 is entirely contained in one
                return 4. / 3. * np.pi * r1 ** 3
            vol = (np.pi * (r1 + r2 - d) ** 2 * (d ** 2 + (2 * d * r1 - 3 * r1 ** 2 + 2 * d * r2 - 3 * r2 ** 2) + 6 * r1 * r2)) / (12 * d)
            return vol
        
        inter = get_intersection(prediction, truth)
        union = ((4/3) * np.pi * (prediction[3]**3) +  (4/3) * np.pi * (truth[3]**3)) -  inter
        return inter / union

    assert type(bboxes) == list
#    bboxes = [box for box in bboxes]
    bboxes = sorted(bboxes, key=lambda x: x[-1], reverse=True)
    bboxes_after_nms = []
    while bboxes:
        chosen_box = bboxes.pop(0)
        bboxes = [ box for box in bboxes if intersection_over_union( np.array(chosen_box[:4]), np.array(box[:4]) ) < iou_threshold ]
        bboxes_after_nms.append(chosen_box)
    return sorted(bboxes_after_nms, key=lambda x: x[-1], reverse=True)

#non_max_suppression(bboxes=boxes, iou_threshold=0.5)

In [ ]:
import Data.IO as  dio
from Volume.Selection import ConnectedComponents2Spheres
import numpy as np

t, v2m = dio.read_nii_from_file ("/path/to/P0010_t_1.nii.gz")

spheres = ConnectedComponents2Spheres (t, v2m)
print(spheres)
spheres[..., :3] = (np.round(apply_transormation(spheres[..., :3], np.linalg.inv(v2m))).astype(np.int32) )/ 48
spheres[..., 3:] = (spheres[..., 3:] / (size[0]/dim[0])) / 48
print(spheres)
#print(spheres[..., 3:])
# transfer to Voxel space
#spheres = trans2vox(spheres, v2m)
#print(spheres)
# go back to metric space
targets = [np.zeros((3, S, S, S, 4 + 0 + 1), dtype='float32') for S in SS]
targets = getBoundingSpheres(spheres, targets, anchors, SS)


### K-means algorithm

In [ ]:
for _ in loader:
    pass

In [ ]:
import json
import os
import math
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns; sns.set()  # for plot styling
from sklearn.cluster import KMeans

n_clusters = 9
raduis = [0.10416666666666667, 0.0625, 0.14583333333333334, 0.10416666666666667, 0.14583333333333334, 0.14583333333333334, 0.0625, 0.08333333333333333, 0.0625, 0.08333333333333333, 0.041666666666666664, 0.0625, 0.0625, 0.041666666666666664, 0.125, 0.10416666666666667, 0.125, 0.0625, 0.10416666666666667, 0.10416666666666667, 0.20833333333333334, 0.16666666666666666, 0.14583333333333334, 0.10416666666666667, 0.08333333333333333, 0.10416666666666667, 0.10416666666666667, 0.0625, 0.0625, 0.0625, 0.125, 0.041666666666666664, 0.125, 0.041666666666666664, 0.10416666666666667, 0.0625, 0.041666666666666664, 0.16666666666666666, 0.08333333333333333, 0.10416666666666667, 0.10416666666666667, 0.14583333333333334, 0.14583333333333334, 0.10416666666666667, 0.16666666666666666, 0.10416666666666667, 0.125, 0.20833333333333334, 0.20833333333333334, 0.041666666666666664, 0.125, 0.041666666666666664, 0.125, 0.041666666666666664, 0.041666666666666664, 0.125, 0.041666666666666664, 0.041666666666666664, 0.0625, 0.25, 0.10416666666666667, 0.020833333333333332, 0.1875, 0.1875, 0.041666666666666664, 0.08333333333333333, 0.041666666666666664, 0.08333333333333333, 0.0625, 0.10416666666666667, 0.041666666666666664, 0.14583333333333334, 0.08333333333333333, 0.16666666666666666, 0.16666666666666666, 0.08333333333333333, 0.14583333333333334, 0.10416666666666667, 0.20833333333333334, 0.08333333333333333, 0.1875, 0.125, 0.08333333333333333, 0.0625, 0.0625, 0.041666666666666664, 0.10416666666666667, 0.041666666666666664, 0.020833333333333332, 0.020833333333333332, 0.1875, 0.16666666666666666, 0.0625, 0.041666666666666664, 0.08333333333333333, 0.08333333333333333, 0.041666666666666664, 0.0625, 0.041666666666666664, 0.0625, 0.0625, 0.0625, 0.125, 0.0625, 0.0625, 0.0625, 0.0625, 0.041666666666666664, 0.020833333333333332, 0.22916666666666666, 0.10416666666666667, 0.125, 0.041666666666666664, 0.0625, 0.5, 0.10416666666666667, 0.08333333333333333, 0.125, 0.08333333333333333, 0.125, 0.08333333333333333, 0.14583333333333334, 0.08333333333333333, 0.16666666666666666, 0.16666666666666666, 0.08333333333333333, 0.0625, 0.10416666666666667, 0.14583333333333334, 0.3125, 0.0625, 0.10416666666666667, 0.0625, 0.10416666666666667, 0.041666666666666664, 0.125, 0.16666666666666666, 0.08333333333333333, 0.16666666666666666, 0.08333333333333333, 0.10416666666666667, 0.041666666666666664, 0.125, 0.16666666666666666, 0.1875, 0.0625, 0.08333333333333333, 0.08333333333333333, 0.08333333333333333, 0.10416666666666667, 0.10416666666666667, 0.08333333333333333, 0.041666666666666664, 0.125, 0.0625, 0.020833333333333332, 0.0625, 0.020833333333333332, 0.0625, 0.0625, 0.0625, 0.08333333333333333, 0.10416666666666667, 0.0625, 0.041666666666666664, 0.0625, 0.0625, 0.10416666666666667, 0.0625, 0.08333333333333333, 0.1875, 0.0625, 0.10416666666666667, 0.10416666666666667, 0.08333333333333333, 0.16666666666666666, 0.22916666666666666, 0.0625, 0.08333333333333333, 0.10416666666666667, 0.14583333333333334, 0.08333333333333333, 0.14583333333333334, 0.08333333333333333, 0.10416666666666667, 0.10416666666666667, 0.14583333333333334, 0.08333333333333333, 0.10416666666666667, 0.14583333333333334, 0.08333333333333333, 0.2708333333333333, 0.08333333333333333, 0.0625, 0.0625, 0.0625, 0.1875, 0.08333333333333333, 0.10416666666666667, 0.08333333333333333, 0.08333333333333333, 0.0625, 0.08333333333333333, 0.3958333333333333, 0.0625, 0.020833333333333332, 0.041666666666666664, 0.14583333333333334, 0.0625, 0.041666666666666664, 0.14583333333333334, 0.08333333333333333, 0.041666666666666664, 0.08333333333333333, 0.125, 0.125, 0.08333333333333333, 0.14583333333333334, 0.08333333333333333, 0.14583333333333334, 0.08333333333333333, 0.0625, 0.10416666666666667, 0.125, 0.25, 0.1875, 0.10416666666666667, 0.10416666666666667, 0.20833333333333334, 0.08333333333333333, 0.16666666666666666, 0.125, 0.020833333333333332, 0.125, 0.0625, 0.0625, 0.125, 0.020833333333333332, 0.020833333333333332, 0.0625, 0.041666666666666664, 0.0625, 0.0625, 0.08333333333333333, 0.10416666666666667, 0.041666666666666664, 0.041666666666666664, 0.0625, 0.10416666666666667, ]
raduis = np.asarray(raduis).reshape(-1, 1)
     
#x=x.transpose()
##########################################   K- Means
##########################################

kmeans3 = KMeans(n_clusters=n_clusters)
kmeans3.fit(raduis)
y_kmeans3 = kmeans3.predict(raduis)


In [ ]:
ax = plt.scatter(y_kmeans3, y='label', s=50, color='white', edgecolor='black')

plt.scatter(x='mse', y='label', s=50, color='white', ax=ax, edgecolor='red')
#plt.scatter(kmeans.cluster_centers_.ravel(), [0.5]*len(kmeans.cluster_centers_), s=100, color='green', marker='*')


In [ ]:
centers3 = kmeans3.cluster_centers_

yolo_anchor_average=[]
for ind in range (n_clusters):
    yolo_anchor_average.append(np.mean(raduis[y_kmeans3==ind],axis=0))
yolo_anchor_average=np.array(yolo_anchor_average)
print(yolo_anchor_average)
plt.scatter(raduis[:, 0],y_kmeans3, s=2, cmap='viridis')
#plt.scatter(yolo_anchor_average[:, 0], c='red', s=50);


In [ ]:
Anchors = [[0.44791667, 0.27083333, 0.2], 
           [0.15456989,  0.125 , 0.10416667 ], 
           [0.08333333, 0.0625,  0.03658537]]
           
Anchors = [
    [(0.28, 0.22), (0.38, 0.48), (0.9, 0.78)],
    [(0.07, 0.15), (0.15, 0.11), (0.14, 0.29)],
    [(0.02, 0.03), (0.04, 0.07), (0.08, 0.06)],
]  # Note these have been rescaled to be between [0, 1]


## Split patients

In [ ]:
# generate splits
import random, os
#pats = ["P0001", "P0006", "P0011", "P0016", "P0021", "P0026", "P0031", "P0036", "P0041", "P0046", "P0051", "P0056", "P0061", "P0066", "P0071", "P0076", "P0081", "P0086", "P0091", "P0096", "P0101", "P0106", "P0111", "P0116", "P0121", "P0126", "P0131", "P0002", "P0007", "P0012", "P0017", "P0022", "P0027", "P0032", "P0037", "P0042", "P0047", "P0052", "P0057", "P0062", "P0067", "P0072", "P0077", "P0082", "P0087", "P0092", "P0097", "P0102", "P0107", "P0112", "P0117", "P0122", "P0127", "P0132", "P0003", "P0008", "P0013", "P0018", "P0023", "P0028", "P0033", "P0038", "P0043", "P0048", "P0053", "P0058", "P0063", "P0068", "P0073", "P0078", "P0083", "P0088", "P0093", "P0098", "P0103", "P0108", "P0113", "P0118", "P0123", "P0128", "P0004", "P0009", "P0014", "P0019", "P0024", "P0029", "P0034", "P0039", "P0044", "P0049", "P0054", "P0059", "P0064", "P0069", "P0074", "P0079", "P0084", "P0089", "P0094", "P0099", "P0104", "P0109", "P0114", "P0119", "P0124", "P0129", "P0005", "P0010", "P0015", "P0020", "P0025", "P0030", "P0035", "P0040", "P0045", "P0050", "P0055", "P0060", "P0065", "P0070", "P0075", "P0080", "P0085", "P0090", "P0095", "P0100", "P0105", "P0110", "P0115", "P0120", "P0125", "P0130"]
pats = ["P0135", "P0138", "P0141", "P0144", "P0147", "P0150", "P0153", "P0156", "P0160", "P0163", "P0166", "P0169", "P0172", "P0175", "P0178", "P0181", "P0184", "P0188", "P0194", "P0197", "P0200", "P0204", "P0207", "P0133", "P0136", "P0139", "P0142", "P0145", "P0148", "P0151", "P0154", "P0157", "P0161", "P0164", "P0167", "P0170", "P0173", "P0176", "P0179", "P0182", "P0185", "P0189", "P0195", "P0198", "P0201", "P0205", "P0210", "P0134", "P0137", "P0140", "P0143", "P0146", "P0149", "P0152", "P0155", "P0158", "P0162", "P0165", "P0168", "P0171", "P0174", "P0177", "P0180", "P0183", "P0187", "P0190", "P0196", "P0199", "P0203", "P0206"]
pats = [os.path.join("/path/to/Data/ADAM", pat)for pat in pats]
random.shuffle(pats)

#select splits
split1 = random.sample(pats, 14)
pats = [item for item in pats if item not in split1]
random.shuffle(pats)

split2 = random.sample(pats, 14)
pats = [item for item in pats if item not in split2]
random.shuffle(pats)

split3 = random.sample(pats, 14)
pats = [item for item in pats if item not in split3]
random.shuffle(pats)

split4 = random.sample(pats, 14)
pats = [item for item in pats if item not in split4]
random.shuffle(pats)

split5 = random.sample(pats, 14)
pats = [item for item in pats if item not in split5]
random.shuffle(pats)

print(len(split1), len(split2), len(split3), len(split4), len(split5))
print(len(pats))

## Test met2vox

In [ ]:
import Data.IO as dio
import numpy as np
from nibabel.affines import apply_affine

vol, vox2met = dio.read_nii_from_file("detected_P0001_v_1.nii.gz")
center_mm = [[-15.725, -4.227, -2.727],
             [-19.135, 2.494, 1.113]
            ]

center_vox = apply_affine(np.linalg.inv(vox2met), center_mm)
print(center_vox)
real_pt = apply_affine(vox2met, center_vox)
print(real_pt)

# Test data loading

In [ ]:
import Data.IO as dio

p = '/path/to/Data/CHRU/P0019'

pat_db = dio.read_patient_data_base([p], volume='noskull volume', normalize=None, log=True, workers=10)


In [ ]:
dio.save_nii_to_file('/path/to/0Work/Experiments/5Folds_Nancy/Fold1/Predictions/200_epochs/F0/P0019.nii.gz', 
                     pat_db[0]['data'], 
                     pat_db[0]['affine'])

# Test batch sampling

In [ ]:
import json, os, random, torch
import numpy as np
import importlib

import Data.IO as dio
from Data.Generators import generateTransforms, getPatches

import Dataset
from tqdm import tqdm

In [ ]:
train_dir = "/path/to/0Work/Experiments/Test_Fold1/"
with open(os.path.join(train_dir, 'ndl_config.json'),'r') as f:
    config = json.load(f)

_, _, patients = dio.readSplit(config['split_file'])
patients = sorted(patients)[:1]


In [ ]:
train_db = dio.add_points_to_patient_data(dio.read_patient_data_base(pat_list = patients,
                                                             volume = config['volume'],
                                                             normalize = config['normalize'],
                                                             vessel = None,
                                                             loadtruthFromFile = None,
                                                             label = "training",
                                                             workers = 12
                                                            ), 
                                      config['negative patch centers'],
                                      nb = None)


In [ ]:
loaders, iterations = Dataset.getDataloaders(train_db = train_db, 
                                     valid_db = None, 
                                     config = config,
                                     shuffle_train = True,
                                     shuffle_val = False,
                                     refined = True,
                                     dual = False,
                                     balancedbatch = True,
                                    )

In [ ]:
for epoch in range(10):
    for x, y in loaders['train']:
        pos, neg = 0, 0
        for patch in y:
            if patch.sum()>0:
                pos +=1
            else :
                neg +=1
        print(pos, neg)
    print("-------")
